In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_curve, auc, precision_recall_curve
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import warnings
from sklearn.feature_selection import SelectKBest, chi2, RFE
import shap
import plotly.express as px
import plotly.graph_objs as go
from plotly.subplots import make_subplots
import pickle
import os

warnings.filterwarnings('ignore')

# Create output directory for visualizations
if not os.path.exists('visualizations'):
    os.makedirs('visualizations')

class PhishingModelTrainer:
    def __init__(self, data_path='new_approach.csv'):
        """Initialize the phishing detection model trainer."""
        self.data_path = data_path
        self.df = None
        self.X = None
        self.y = None
        self.X_train = None
        self.X_test = None
        self.y_train = None
        self.y_test = None
        self.models = {}
        self.best_model = None
        self.scaler = StandardScaler()
        
    def load_data(self):
        """Load and prepare the dataset."""
        print("Loading dataset...")
        self.df = pd.read_csv(self.data_path)
        self.X = self.df.iloc[:, :-1]
        self.y = self.df.iloc[:, -1]

        # Relabel -1 to 0
        self.y = self.y.replace(-1, 0)

        # Dataset info
        print(f"Dataset shape: {self.df.shape}")
        print(f"Number of features: {self.X.shape[1]}")
        print(f"Class distribution:\n{self.y.value_counts()}")

        # Split the data
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42, stratify=self.y
        )

        # Scale the features
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)

        print("Data loading completed.\n")

        
    def visualize_data_distribution(self):
        """Create visualizations for data distribution."""
        print("Creating data distribution visualizations...")
        
        # 1. Class distribution pie chart
        plt.figure(figsize=(10, 8))
        plt.subplot(2, 2, 1)
        self.y.value_counts().plot(kind='pie', autopct='%1.1f%%', colors=['#ff9999', '#66b3ff'])
        plt.title('Class Distribution')
        plt.ylabel('')
        
        # 2. Feature correlation heatmap (using top 10 features for clarity)
        plt.subplot(2, 2, 2)
        correlation_matrix = self.df.iloc[:, :10].corr()  # Top 10 features for clarity
        sns.heatmap(correlation_matrix, cmap='coolwarm', center=0)
        plt.title('Feature Correlation Heatmap')
        
        # 3. Feature distribution
        plt.subplot(2, 2, 3)
        feature_importance_data = []
        for col in self.X.columns[:10]:  # Show top 10 features
            feature_importance_data.append({
                'feature': col,
                'mean': self.X[col].mean(),
                'std': self.X[col].std()
            })
        feature_df = pd.DataFrame(feature_importance_data)
        sns.barplot(data=feature_df, x='feature', y='mean')
        plt.xticks(rotation=45)
        plt.title('Feature Mean Values (Top 10)')
        
        # 4. Missing values check
        plt.subplot(2, 2, 4)
        missing_values = self.df.isnull().sum()
        if missing_values.sum() > 0:
            missing_values[missing_values > 0].plot(kind='bar')
            plt.title('Missing Values by Feature')
        else:
            plt.text(0.5, 0.5, 'No missing values', ha='center', va='center', transform=plt.gca().transAxes)
            plt.title('Missing Values Check')
        
        plt.tight_layout()
        plt.savefig('visualizations/data_distribution.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # 5. Interactive feature distribution plot using separate plots
        # Class distribution pie chart
        fig1 = px.pie(values=self.y.value_counts().values, 
                    names=self.y.value_counts().index,
                    title='Class Distribution',
                    color_discrete_sequence=['#ff9999', '#66b3ff'])
        fig1.write_html('visualizations/interactive_class_distribution.html')
        
        # Feature box plots
        fig2 = go.Figure()
        for col in self.X.columns[:5]:  # First 5 features
            fig2.add_trace(go.Box(y=self.X[col], name=col))
        fig2.update_layout(title='Feature Values Distribution', height=600, width=1000)
        fig2.write_html('visualizations/interactive_feature_distribution.html')
        
        # Feature variance bar chart
        variances = self.X.var().sort_values(ascending=False)[:10]
        fig3 = px.bar(x=variances.index, y=variances.values, 
                    title='Top 10 Features by Variance',
                    labels={'x': 'Features', 'y': 'Variance'})
        fig3.update_layout(height=600, width=1000)
        fig3.write_html('visualizations/interactive_feature_variance.html')
        
        # Feature heatmap
        fig4 = go.Figure(data=go.Heatmap(
            z=correlation_matrix.values,
            x=correlation_matrix.columns,
            y=correlation_matrix.columns,
            colorscale='RdBu'
        ))
        fig4.update_layout(title='Feature Correlation Heatmap', height=800, width=800)
        fig4.write_html('visualizations/interactive_correlation_heatmap.html')

        
        
        print("Data distribution visualizations saved.\n")
        
    def feature_selection(self):
        """Perform feature selection and visualize important features."""
        print("Performing feature selection...")
        
        # Feature importance using Random Forest
        rf = RandomForestClassifier(n_estimators=100, random_state=42)
        rf.fit(self.X_train, self.y_train)
        
        feature_importance = pd.DataFrame({
            'feature': self.X.columns,
            'importance': rf.feature_importances_
        }).sort_values('importance', ascending=False)
        
        # Plot feature importance
        plt.figure(figsize=(12, 8))
        plt.subplot(2, 1, 1)
        sns.barplot(data=feature_importance.head(15), x='importance', y='feature')
        plt.title('Top 15 Features by Random Forest Importance')
        
        # Cumulative feature importance
        plt.subplot(2, 1, 2)
        cumulative_importance = np.cumsum(feature_importance['importance'])
        plt.plot(range(1, len(cumulative_importance) + 1), cumulative_importance)
        plt.axhline(y=0.95, color='r', linestyle='--', label='95% Variance Explained')
        plt.xlabel('Number of Features')
        plt.ylabel('Cumulative Importance')
        plt.title('Cumulative Feature Importance')
        plt.legend()
        
        plt.tight_layout()
        plt.savefig('visualizations/feature_importance.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Interactive feature importance
        fig = px.bar(feature_importance.head(20), x='importance', y='feature', 
                     orientation='h', title='Top 20 Features by Importance')
        fig.update_layout(height=800, width=1000)
        fig.write_html('visualizations/interactive_feature_importance.html')
        
        return feature_importance
        
    def train_models(self):
        """Train multiple models and compare their performance."""
        print("Training models...")
        
        # Define models
        self.models = {
            'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42),
            'XGBoost': XGBClassifier(n_estimators=200, learning_rate=0.01, max_depth=8, random_state=42),
            'LightGBM': LGBMClassifier(n_estimators=200, learning_rate=0.01, max_depth=8, random_state=42),
            'CatBoost': CatBoostClassifier(iterations=200, learning_rate=0.01, depth=8, random_state=42, verbose=False),
            'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, learning_rate=0.01, max_depth=8, random_state=42),
            'Extra Trees': ExtraTreesClassifier(n_estimators=200, max_depth=15, random_state=42)
        }
        
        # Train and evaluate each model
        results = []
        for name, model in self.models.items():
            print(f"Training {name}...")
            
            # Train model
            model.fit(self.X_train, self.y_train)
            
            # Make predictions
            y_pred = model.predict(self.X_test)
            y_pred_proba = model.predict_proba(self.X_test)[:, 1]
            
            # Calculate metrics
            accuracy = accuracy_score(self.y_test, y_pred)
            
            # Store results
            results.append({
                'Model': name,
                'Accuracy': accuracy,
                'y_pred': y_pred,
                'y_pred_proba': y_pred_proba
            })
            
            print(f"{name} - Accuracy: {accuracy:.4f}\n")
        
        return results
    
    def create_ensemble_model(self):
        """Create an optimized ensemble model."""
        print("Creating ensemble model...")
        
        # Base models with tuned parameters
        rf = RandomForestClassifier(n_estimators=200, max_depth=15, random_state=42)
        xgb = XGBClassifier(n_estimators=200, learning_rate=0.01, max_depth=8, random_state=42)
        lgbm = LGBMClassifier(n_estimators=200, learning_rate=0.01, max_depth=8, random_state=42)
        
        # Create voting ensemble
        ensemble = VotingClassifier(
            estimators=[('rf', rf), ('xgb', xgb), ('lgbm', lgbm)],
            voting='soft'
        )
        
        # Train ensemble
        ensemble.fit(self.X_train, self.y_train)
        
        return ensemble
    
    def visualize_model_comparison(self, results):
        """Visualize model performance comparison."""
        print("Creating model comparison visualizations...")
        
        # Extract accuracy scores
        accuracies = [result['Accuracy'] for result in results]
        model_names = [result['Model'] for result in results]
        
        # 1. Model accuracy comparison
        plt.figure(figsize=(15, 10))
        
        # Bar plot
        plt.subplot(2, 2, 1)
        bars = plt.bar(model_names, accuracies, color='skyblue', edgecolor='navy')
        plt.title('Model Accuracy Comparison')
        plt.ylabel('Accuracy')
        plt.xticks(rotation=45)
        
        # Add value labels on bars
        for bar, acc in zip(bars, accuracies):
            plt.text(bar.get_x() + bar.get_width()/2., bar.get_height(),
                    f'{acc:.3f}', ha='center', va='bottom')
        
        # ROC curves
        plt.subplot(2, 2, 2)
        for result in results:
            fpr, tpr, _ = roc_curve(self.y_test, result['y_pred_proba'])
            auc_score = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"{result['Model']} (AUC = {auc_score:.3f})")
        
        plt.plot([0, 1], [0, 1], 'k--')
        plt.xlabel('False Positive Rate')
        plt.ylabel('True Positive Rate')
        plt.title('ROC Curves Comparison')
        plt.legend()
        
        # Precision-Recall curves
        plt.subplot(2, 2, 3)
        for result in results:
            precision, recall, _ = precision_recall_curve(self.y_test, result['y_pred_proba'])
            plt.plot(recall, precision, label=result['Model'])
        
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title('Precision-Recall Curves')
        plt.legend()
        
        # Confusion matrix heatmap
        plt.subplot(2, 2, 4)
        best_result = max(results, key=lambda x: x['Accuracy'])
        cm = confusion_matrix(self.y_test, best_result['y_pred'])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix - {best_result["Model"]}')
        plt.xlabel('Predicted')
        plt.ylabel('Actual')
        
        plt.tight_layout()
        plt.savefig('visualizations/model_comparison.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Interactive model comparison
        fig = go.Figure()
        
        # Add accuracy bars
        fig.add_trace(go.Bar(
            x=model_names,
            y=accuracies,
            name='Accuracy',
            text=[f'{acc:.3f}' for acc in accuracies],
            textposition='outside'
        ))
        
        fig.update_layout(
            title='Model Performance Comparison',
            xaxis_title='Models',
            yaxis_title='Accuracy',
            height=600,
            width=1000
        )
        
        fig.write_html('visualizations/interactive_model_comparison.html')
        
    def visualize_ensemble_performance(self, ensemble, results):
        """Visualize ensemble model performance."""
        print("Creating ensemble model visualizations...")
        
        # Get ensemble predictions
        y_pred_ensemble = ensemble.predict(self.X_test)
        y_pred_proba_ensemble = ensemble.predict_proba(self.X_test)[:, 1]
        ensemble_accuracy = accuracy_score(self.y_test, y_pred_ensemble)
        
        # Add ensemble to results for comparison
        results.append({
            'Model': 'Ensemble',
            'Accuracy': ensemble_accuracy,
            'y_pred': y_pred_ensemble,
            'y_pred_proba': y_pred_proba_ensemble
        })
        
        # Re-visualize with ensemble included
        self.visualize_model_comparison(results)
        
        # SHAP values for ensemble
        explainer = shap.KernelExplainer(ensemble.predict_proba, 
                                        shap.sample(self.X_train, 100))
        shap_values = explainer.shap_values(self.X_test[:100])
        
        # SHAP summary plot
        plt.figure(figsize=(12, 8))
        shap.summary_plot(shap_values[1], self.X_test[:100], show=False)
        plt.tight_layout()
        plt.savefig('visualizations/shap_summary.png', dpi=300, bbox_inches='tight')
        plt.close()
        
        # Feature importance comparison across models
        fig, ax = plt.subplots(figsize=(12, 8))
        importance_data = []
        
        for name, model in self.models.items():
            if hasattr(model, 'feature_importances_'):
                importance_data.append({
                    'Model': name,
                    'Feature': self.X.columns[np.argsort(model.feature_importances_)[-1]],
                    'Importance': model.feature_importances_[np.argsort(model.feature_importances_)[-1]]
                })
        
        importance_df = pd.DataFrame(importance_data)
        sns.barplot(data=importance_df, x='Model', y='Importance', hue='Feature', ax=ax)
        plt.title('Top Feature by Model')
        plt.xticks(rotation=45)
        plt.tight_layout()
        plt.savefig('visualizations/top_features_by_model.png', dpi=300, bbox_inches='tight')
        plt.close()
        
    def save_model(self, model, filename='phishing_detection_model.pkl'):
        """Save the trained model."""
        with open(filename, 'wb') as f:
            pickle.dump(model, f)
        print(f"Model saved to {filename}")
        
    def generate_report(self, results, ensemble_accuracy):
        """Generate comprehensive performance report."""
        report_file = 'visualizations/performance_report.txt'
        
        with open(report_file, 'w') as f:
            f.write("PHISHING DETECTION MODEL PERFORMANCE REPORT\n")
            f.write("=========================================\n\n")
            
            f.write(f"Dataset: {self.data_path}\n")
            f.write(f"Total samples: {len(self.df)}\n")
            f.write(f"Number of features: {self.X.shape[1]}\n")
            f.write(f"Class distribution:\n{self.y.value_counts()}\n\n")
            
            f.write("MODEL PERFORMANCE SUMMARY\n")
            f.write("------------------------\n")
            for result in results:
                f.write(f"{result['Model']: <20} Accuracy: {result['Accuracy']:.4f}\n")
            
            f.write("\nBEST PERFORMING MODELS\n")
            f.write("---------------------\n")
            sorted_results = sorted(results, key=lambda x: x['Accuracy'], reverse=True)
            for i, result in enumerate(sorted_results[:3], 1):
                f.write(f"{i}. {result['Model']: <20} Accuracy: {result['Accuracy']:.4f}\n")
                
            f.write("\nENSEMBLE MODEL PERFORMANCE\n")
            f.write("------------------------\n")
            f.write(f"Ensemble Accuracy: {ensemble_accuracy:.4f}\n")
            
            f.write("\nFEATURE IMPORTANCE (Top 10)\n")
            f.write("-------------------------\n")
            rf = self.models['Random Forest']
            feature_importance = pd.DataFrame({
                'feature': self.X.columns,
                'importance': rf.feature_importances_
            }).sort_values('importance', ascending=False)
            
            for idx, row in feature_importance.head(10).iterrows():
                f.write(f"{row['feature']: <30} {row['importance']:.4f}\n")
                
        print(f"Performance report saved to {report_file}")
        
    def run_complete_pipeline(self):
        """Run the complete training and evaluation pipeline."""
        print("Starting phishing detection model pipeline...\n")
        
        # Load data
        self.load_data()
        
        # Visualize data distribution
        # self.visualize_data_distribution()
        
        # Feature selection
        feature_importance = self.feature_selection()
        
        # Train models
        results = self.train_models()
        
        # Create ensemble model
        ensemble = self.create_ensemble_model()
        ensemble_accuracy = accuracy_score(self.y_test, ensemble.predict(self.X_test))
        
        # Visualize model comparison
        self.visualize_model_comparison(results)
        
        # Visualize ensemble performance
        self.visualize_ensemble_performance(ensemble, results)
        
        # Save best model
        best_single_model = max(results[:-1], key=lambda x: x['Accuracy'])['Model']
        self.save_model(ensemble, 'phishing_ensemble_model.pkl')
        self.save_model(self.models[best_single_model], f'phishing_{best_single_model.lower().replace(" ", "_")}_model.pkl')
        
        # Generate report
        self.generate_report(results, ensemble_accuracy)
        
        # Print summary
        print("\nPIPELINE COMPLETED SUCCESSFULLY!")
        print("--------------------------------")
        print(f"Best single model: {best_single_model} (Accuracy: {max(result['Accuracy'] for result in results[:-1]):.4f})")
        print(f"Ensemble model accuracy: {ensemble_accuracy:.4f}")
        print("\nVisualization files saved in 'visualizations/' directory:")
        print("- data_distribution.png")
        print("- feature_importance.png")
        print("- model_comparison.png")
        print("- shap_summary.png")
        print("- top_features_by_model.png")
        print("- interactive_data_analysis.html")
        print("- interactive_feature_importance.html")
        print("- interactive_model_comparison.html")
        print("- performance_report.txt")
        print("\nModels saved:")
        print("- phishing_ensemble_model.pkl")
        print(f"- phishing_{best_single_model.lower().replace(' ', '_')}_model.pkl")
        print("\nDone!")

# Example usage
if __name__ == "__main__":
    trainer = PhishingModelTrainer('new_approach.csv')
    trainer.run_complete_pipeline()

Starting phishing detection model pipeline...

Loading dataset...
Dataset shape: (59455, 35)
Number of features: 34
Class distribution:
34
0    32309
1    27146
Name: count, dtype: int64
Data loading completed.

Performing feature selection...
Training models...
Training Random Forest...
Random Forest - Accuracy: 0.8780

Training XGBoost...
XGBoost - Accuracy: 0.8597

Training LightGBM...
[LightGBM] [Info] Number of positive: 21717, number of negative: 25847
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.003971 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3663
[LightGBM] [Info] Number of data points in the train set: 47564, number of used features: 28
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.456585 -> initscore=-0.174099
[LightGBM] [Info] Start training from score -0.174099
LightGBM - Accuracy: 0.8432

Training CatBoost...


  0%|          | 0/100 [00:00<?, ?it/s]

AssertionError: The shape of the shap_values matrix does not match the shape of the provided data matrix.

<Figure size 1200x800 with 0 Axes>